# Music Annotation Project
## Notebook 74: Combined Mask Authority, Annotation Distance and Print-Loss Audit

**Runtime: CPU. Read-only diagnostic. Run all.**

This revision processes both cohorts in one run:

- **HITL annotated-only**: resolves the 151 reviewed masks and confidence maps through HITL audit records, and audits the latest complete Notebook 72 outputs.
- **Paired development**: resolves clean pages, aligned annotated pages, undilated corrected masks and frozen V7.5/V7.8 cached arrays. Clean truth is used only for post-hoc labels and scoring.

The cohorts remain explicitly labelled. The notebook does not pool observable `print_like` evidence with clean-supported genuine print as though they were the same endpoint.

### Primary paired-truth endpoint

For paired pages, genuine-print loss is measured using the frozen project printed-ink method with one-pixel registration tolerance. Distance to **truth-side annotation evidence** is computed from dark observed pixels inside the seed that are not supported by tolerant clean printed ink.

### Main visual outputs

- cohort-specific authority-band charts;
- seed-versus-collar page scatter plots;
- authority-band × annotation-distance heatmaps;
- clean-truth genuine-print-loss heatmaps for paired pages;
- worst-page whole-page contact sheets for each cohort;
- focused loss crops for paired pages;
- a combined dashboard that keeps the two evidence types visibly separate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib seaborn pillow

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile, gc
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
RUN_HITL=True
RUN_PAIRED=True

# HITL / Notebook 72 lineage
NOTEBOOK72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
NOTEBOOK72_RUN_OVERRIDE=None
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'

# Paired development lineage
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V75_RUN_OVERRIDE=None
DEVELOPMENT_MANIFEST=ROOT/'development_score_manifest.csv'
CLEAN_DIR=ROOT/'source'/'clean'
ALIGNED_DIR=ROOT/'aligned'
DEVELOPMENT_SEED_DIRS=[ROOT/'masks'/'corrected']

OUTPUT_ROOT=ROOT/'notebook74_combined_mask_annotation_distance_audit'
CONFIDENCE_LEVELS=[0.50,0.80,0.95]
DARK_THRESHOLD=190
CHANGE_THRESHOLD=8
PRINT_LIKE_VALUE_THRESHOLD=200
DISTANCE_BINS=[-0.001,1,2,4,8,16,32,np.inf]
MAX_CONTACT_PAGES=16
MAX_FOCUS_CROPS=36
CROP_MARGIN=80

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook74_combined'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; PARTS=TABLES/'pixel_parts'; PAGES=OUT/'pages'; VISUAL=OUT/'visuals'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,PARTS,PAGES,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Shared helpers and authoritative source resolution

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def require(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp');t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str));t.replace(path)
def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE);require(a is not None,f'Unreadable image: {path}');return a
def imap(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def distance_to_core(core):
    if not np.any(core):return np.full(core.shape,np.nan,np.float32)
    return cv2.distanceTransform((~core).astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
def printed_ink(clean):return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant_clean_print(clean):return cv2.dilate(printed_ink(clean).astype(np.uint8),np.ones((3,3),np.uint8))>0
def band(values):
    labels=[f'{int(max(a,0))}-{"inf" if np.isinf(b) else int(b)}' for a,b in zip(DISTANCE_BINS[:-1],DISTANCE_BINS[1:])]
    return pd.cut(values,DISTANCE_BINS,labels=labels,include_lowest=True,right=False)
def audit_records(folder,label):
    out=[];bad=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:
            v=json.loads(p.read_text());require(isinstance(v,dict),'audit root not object');v['_audit_path']=str(p);v['_audit_type']=label;out.append(v)
        except Exception as e:bad.append({'type':label,'path':str(p),'error':repr(e)})
    return out,bad
def required_path(record,key):
    p=Path(str(record.get(key,'')));require(str(p) and p.exists(),f'Audit {record.get("_audit_path")} missing {key}: {p}');return p

def geometry(seed):
    k1=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3));k2=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    d1=cv2.dilate(seed.astype(np.uint8),k1)>0;d2=cv2.dilate(seed.astype(np.uint8),k2)>0
    return d1,d2,d1&~seed,d2&~d1

review_audits,rbad=audit_records(REVIEW_AUDIT_DIR,'review');draft_audits,dbad=audit_records(DRAFT_AUDIT_DIR,'draft')
if RUN_HITL and (rbad or dbad):
    pd.DataFrame(rbad+dbad).to_csv(AUDIT/'hitl_audit_parse_failures.csv',index=False);raise RuntimeError('Unreadable HITL audit records.')

## 3. Build a unified, explicitly role-labelled page roster

In [ ]:
roster=[]
# HITL pages from latest complete Notebook 72 run.
if RUN_HITL:
    req=['prediction_lock.json','notebook72_verdict.json','inputs/runtime_roster.csv','tables/page_output_hashes.csv','pages']
    N72=Path(NOTEBOOK72_RUN_OVERRIDE) if NOTEBOOK72_RUN_OVERRIDE else latest(NOTEBOOK72_ROOT,req)
    require(N72 is not None,'No complete Notebook 72 run found')
    lock=json.loads((N72/'prediction_lock.json').read_text());verdict=json.loads((N72/'notebook72_verdict.json').read_text())
    require(lock.get('state')=='CLOSED','Notebook 72 lock is not closed');require(verdict.get('first_failed_gate') is None,'Notebook 72 failed a gate')
    rr=pd.read_csv(N72/'inputs/runtime_roster.csv',dtype=str).fillna('')
    for _,r in rr.iterrows():
        source=Path(str(r.source_path));require(source.exists(),f'Missing HITL source {source}');m=md5_file(source)
        rev=[x for x in review_audits if str(x.get('source_md5','')).lower()==m.lower()];dra=[x for x in draft_audits if str(x.get('source_md5','')).lower()==m.lower()]
        require(len(rev)==1,f'{source.name}: expected exactly one review audit, got {len(rev)}');require(len(dra)<=1,f'{source.name}: duplicate draft audits')
        seed_path=required_path(rev[0],'corrected_mask_output');conf_path=required_path(dra[0],'confidence_map') if dra else None
        key=f'hitl__{r.population}__{r.page}'
        roster.append({'page_key':key,'page':str(r.page),'population':'HITL_ANNOTATED_ONLY','source_group':str(r.population),'observed_path':str(N72/'pages'/f'{r.population}__{r.page}'/'observed.png'),'corrected_path':str(N72/'pages'/f'{r.population}__{r.page}'/'corrected.png'),'prepared_path':str(N72/'pages'/f'{r.population}__{r.page}'/'prepared_mask.png'),'reason_path':str(N72/'pages'/f'{r.population}__{r.page}'/'reason_map.png'),'seed_path':str(seed_path),'confidence_path':str(conf_path) if conf_path else None,'clean_path':None,'cache_path':None,'correction_source':'V8.8_NOTEBOOK72'})

# Paired development pages from frozen V7.5 caches.
if RUN_PAIRED:
    V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
    require(V75 is not None,'No complete V7.5 run found')
    status=pd.read_csv(V75/'tables/cache_status.csv');cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
    clean=imap(CLEAN_DIR);obs=imap(ALIGNED_DIR);seeds={}
    for d in DEVELOPMENT_SEED_DIRS:seeds.update(imap(d))
    manifest=pd.read_csv(DEVELOPMENT_MANIFEST,dtype=str).fillna('');require({'page','source_group'}<=set(manifest),'Development manifest lacks page/source_group')
    meta=manifest.set_index(manifest.page.astype(str))
    pages=sorted(set(manifest.page.astype(str))&set(cache)&set(clean)&set(obs)&set(seeds))
    require(pages,'No complete paired development pages found')
    missing=sorted(set(manifest.page.astype(str))-set(pages));require(not missing,f'Incomplete paired development roster: {missing}')
    for page in pages:
        roster.append({'page_key':f'paired__{page}','page':page,'population':'PAIRED_DEVELOPMENT','source_group':str(meta.loc[page,'source_group']),'observed_path':str(obs[page]),'corrected_path':None,'prepared_path':None,'reason_path':None,'seed_path':str(seeds[page]),'confidence_path':None,'clean_path':str(clean[page]),'cache_path':str(cache[page]),'correction_source':'FROZEN_V7.8_GEOMETRY_T80'})

roster=pd.DataFrame(roster);require(not roster.page_key.duplicated().any(),'Duplicate page keys');roster.to_csv(INPUTS/'combined_roster.csv',index=False)
print(roster.groupby(['population','correction_source']).size())

## 4. Page loader and integrity checks

In [ ]:
def load_page(r):
    obs=read_gray(r.observed_path);seed=read_gray(r.seed_path)>0;clean=read_gray(r.clean_path) if pd.notna(r.clean_path) and r.clean_path else None
    confidence=None
    if pd.notna(r.confidence_path) and r.confidence_path:
        confidence=np.load(r.confidence_path).astype(np.float32);confidence=confidence/255.0 if np.nanmax(confidence)>1.00001 else confidence
    if r.population=='HITL_ANNOTATED_ONLY':
        corrected=read_gray(r.corrected_path);prepared=read_gray(r.prepared_path)>0;reason=read_gray(r.reason_path)
        route=np.select([reason==1,reason==2,reason==3,reason==4],['V78_PRESERVE','STRICT_PRINT','M5_REMOVE','SAFE_REMOVE'],default='OTHER')
    else:
        with np.load(r.cache_path) as z:
            need=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous'];miss=[x for x in need if x not in z.files];require(not miss,f'{r.page_key}: cache missing {miss}')
            prepared=z['mask']>0;corrected=z['geometry_t80'];pres=z['g80_preserve']>0;rem=z['g80_remove']>0;amb=z['g80_ambiguous']>0
        route=np.select([pres,rem,amb],['V78_PRESERVE','V78_REMOVE','V78_AMBIGUOUS'],default='OTHER')
    require(obs.shape==seed.shape==prepared.shape==corrected.shape,f'{r.page_key}: shape mismatch')
    if clean is not None:require(clean.shape==obs.shape,f'{r.page_key}: clean shape mismatch')
    if confidence is not None:require(confidence.shape==obs.shape,f'{r.page_key}: confidence shape mismatch')
    d1,d2,ring1,ring2=geometry(seed);require(np.all(seed<=prepared),f'{r.page_key}: seed not subset prepared')
    dilation_xor=int(np.logical_xor(prepared,d2).sum())
    return {'obs':obs,'seed':seed,'prepared':prepared,'corrected':corrected,'clean':clean,'confidence':confidence,'ring1':ring1,'ring2':ring2,'route':route,'dilation_xor':dilation_xor}

checks=[]
for _,r in roster.iterrows():
    z=load_page(r);checks.append({'page_key':r.page_key,'population':r.population,'source_group':r.source_group,'seed_pixels':int(z['seed'].sum()),'ring1_pixels':int(z['ring1'].sum()),'ring2_pixels':int(z['ring2'].sum()),'prepared_pixels':int(z['prepared'].sum()),'prepared_vs_exact_dilation2_xor':z['dilation_xor'],'clean_available':z['clean'] is not None,'confidence_available':z['confidence'] is not None});del z;gc.collect()
checks=pd.DataFrame(checks);checks.to_csv(AUDIT/'input_resolution_and_geometry.csv',index=False)
require((checks.prepared_vs_exact_dilation2_xor==0).all(),'At least one prepared mask is not exact dilation-2 of its authoritative seed')
print(checks.groupby('population')[['clean_available','confidence_available']].sum())

## 5. Pixel audit: observable evidence for all pages, clean truth where available

In [ ]:
page_rows=[];cell_rows=[]
for _,r in roster.iterrows():
    z=load_page(r);obs=z['obs'];corrected=z['corrected'];seed=z['seed'];prepared=z['prepared'];ring1=z['ring1'];ring2=z['ring2'];clean=z['clean'];confidence=z['confidence'];route=z['route']
    authority=np.full(obs.shape,'OTHER_PREPARED',object);authority[seed]='SEED';authority[ring1]='RING1';authority[ring2]='RING2'
    delta=corrected.astype(np.int16)-obs.astype(np.int16);changed=prepared&(np.abs(delta)>=CHANGE_THRESHOLD);whitened=prepared&(delta>=CHANGE_THRESHOLD)
    dark_core=seed&(obs<DARK_THRESHOLD);proxy_dist=distance_to_core(dark_core)
    edges=cv2.Canny(obs,70,170)>0;print_like=(obs<PRINT_LIKE_VALUE_THRESHOLD)&edges
    truth_print=np.zeros(obs.shape,bool);truth_ann=np.zeros(obs.shape,bool);truth_dist=np.full(obs.shape,np.nan,np.float32)
    if clean is not None:
        truth_print=tolerant_clean_print(clean);truth_ann=seed&(obs<DARK_THRESHOLD)&~truth_print;truth_dist=distance_to_core(truth_ann)
    ys,xs=np.where(prepared)
    f=pd.DataFrame({'page_key':r.page_key,'page':r.page,'population':r.population,'source_group':r.source_group,'y':ys.astype(np.int32),'x':xs.astype(np.int32),'authority_band':authority[ys,xs].astype(str),'route':route[ys,xs].astype(str),'observed':obs[ys,xs].astype(np.uint8),'signed_change':delta[ys,xs].astype(np.int16),'changed_8plus':changed[ys,xs],'whitened_8plus':whitened[ys,xs],'print_like_observable':print_like[ys,xs],'distance_to_dark_seed_proxy':proxy_dist[ys,xs].astype(np.float32),'truth_print_posthoc':truth_print[ys,xs],'truth_annotation_posthoc':truth_ann[ys,xs],'distance_to_truth_annotation_posthoc':truth_dist[ys,xs].astype(np.float32)})
    f['proxy_distance_band']=band(f.distance_to_dark_seed_proxy);f['truth_distance_band']=band(f.distance_to_truth_annotation_posthoc)
    for t in CONFIDENCE_LEVELS:
        core=prepared&(confidence>=t) if confidence is not None else np.zeros(obs.shape,bool);f[f'distance_to_conf_{int(t*100)}']=distance_to_core(core)[ys,xs]
    f.to_parquet(PARTS/f'{r.page_key}.parquet',index=False)
    page_rows.append({'page_key':r.page_key,'page':r.page,'population':r.population,'source_group':r.source_group,'prepared_pixels':int(prepared.sum()),'seed_pixels':int(seed.sum()),'ring1_pixels':int(ring1.sum()),'ring2_pixels':int(ring2.sum()),'changed_pixels':int(changed.sum()),'whitened_pixels':int(whitened.sum()),'seed_whitened_observable':int((whitened&print_like&seed).sum()),'collar_whitened_observable':int((whitened&print_like&(ring1|ring2)).sum()),'seed_true_print_whitened':int((whitened&truth_print&seed).sum()) if clean is not None else np.nan,'collar_true_print_whitened':int((whitened&truth_print&(ring1|ring2)).sum()) if clean is not None else np.nan})
    distance_col='truth_distance_band' if clean is not None else 'proxy_distance_band';denom_col='truth_print_posthoc' if clean is not None else 'print_like_observable'
    for (a,d),q in f.groupby(['authority_band',distance_col],observed=True):
        denom=q[denom_col].astype(bool);cell_rows.append({'population':r.population,'page_key':r.page_key,'authority_band':a,'distance_band':str(d),'evidence_type':'CLEAN_TRUE_PRINT' if clean is not None else 'OBSERVABLE_PRINT_LIKE','eligible_evidence_pixels':int(denom.sum()),'whitened_evidence_pixels':int((denom&q.whitened_8plus).sum())})
    del z,f,obs,corrected,seed,prepared,ring1,ring2,clean,confidence,route,authority,delta,changed,whitened,dark_core,proxy_dist,edges,print_like,truth_print,truth_ann,truth_dist,ys,xs;gc.collect()
page_df=pd.DataFrame(page_rows);cell_df=pd.DataFrame(cell_rows);page_df.to_csv(TABLES/'page_summary.csv',index=False);cell_df.to_csv(TABLES/'authority_by_distance_by_page.csv',index=False)
pooled=cell_df.groupby(['population','evidence_type','authority_band','distance_band'],as_index=False).agg(eligible_evidence_pixels=('eligible_evidence_pixels','sum'),whitened_evidence_pixels=('whitened_evidence_pixels','sum'));pooled['whitened_rate']=pooled.whitened_evidence_pixels/np.maximum(pooled.eligible_evidence_pixels,1);pooled.to_csv(TABLES/'authority_by_distance_pooled.csv',index=False)
print(page_df.groupby('population')[['changed_pixels','whitened_pixels']].sum())

## 6. Aggregate visual dashboard

In [ ]:
sns.set_theme(style='whitegrid')
# Authority activity by cohort.
rows=[]
for p in PARTS.glob('*.parquet'):
    q=pd.read_parquet(p,columns=['population','authority_band','changed_8plus','whitened_8plus']);rows.append(q.groupby(['population','authority_band'],as_index=False).agg(pixels=('changed_8plus','size'),changed=('changed_8plus','sum'),whitened=('whitened_8plus','sum')))
auth=pd.concat(rows).groupby(['population','authority_band'],as_index=False).sum();auth['changed_rate']=auth.changed/auth.pixels;auth['whitened_rate']=auth.whitened/auth.pixels;auth.to_csv(TABLES/'authority_band_summary.csv',index=False)
fig,axes=plt.subplots(1,len(auth.population.unique()),figsize=(8*len(auth.population.unique()),5),squeeze=False)
for ax,(pop,q) in zip(axes[0],auth.groupby('population')):
    q=q.set_index('authority_band').reindex(['SEED','RING1','RING2','OTHER_PREPARED']).dropna().reset_index();x=np.arange(len(q));ax.bar(x-.18,q.changed_rate,.36,label='changed >=8');ax.bar(x+.18,q.whitened_rate,.36,label='whitened >=8');ax.set_xticks(x,q.authority_band);ax.set_ylim(0,max(.01,q[['changed_rate','whitened_rate']].to_numpy().max()*1.15));ax.set_title(pop);ax.legend()
fig.suptitle('Correction activity by authority band and cohort');plt.tight_layout();plt.savefig(VISUAL/'authority_band_change_rates_by_cohort.png',dpi=180);plt.close(fig)
# Page scatter, separate panels.
fig,axes=plt.subplots(1,len(page_df.population.unique()),figsize=(8*len(page_df.population.unique()),6),squeeze=False)
for ax,(pop,q) in zip(axes[0],page_df.groupby('population')):
    if pop=='PAIRED_DEVELOPMENT':x='seed_true_print_whitened';y='collar_true_print_whitened';title='Clean-supported genuine print loss'
    else:x='seed_whitened_observable';y='collar_whitened_observable';title='Observable dark structural candidate whitening'
    sns.scatterplot(data=q,x=x,y=y,hue='source_group',size='whitened_pixels',sizes=(20,220),alpha=.75,legend=False,ax=ax);mx=max(q[x].max(),q[y].max(),1);ax.plot([0,mx],[0,mx],'k--');ax.set_title(f'{pop}\n{title}');ax.set_xlabel('inside seed');ax.set_ylabel('rings 1+2')
plt.tight_layout();plt.savefig(VISUAL/'seed_vs_collar_by_cohort.png',dpi=180);plt.close(fig)
# One heatmap per evidence type/cohort.
for (pop,evidence),q in pooled.groupby(['population','evidence_type']):
    pv=q.pivot(index='authority_band',columns='distance_band',values='whitened_rate').reindex(['SEED','RING1','RING2','OTHER_PREPARED'])
    plt.figure(figsize=(12,5));sns.heatmap(pv,annot=True,fmt='.3f',cmap='magma',linewidths=.5);plt.title(f'{pop}: {evidence} whitening rate by authority and annotation distance');plt.xlabel('Distance to truth annotation' if evidence=='CLEAN_TRUE_PRINT' else 'Distance to dark seed proxy');plt.tight_layout();plt.savefig(VISUAL/f'heatmap_{pop.lower()}_{evidence.lower()}.png',dpi=190);plt.close()

## 7. Per-page overlays and contact sheets

In [ ]:
COL={'SEED':(0,180,0),'RING1':(0,215,255),'RING2':(0,0,255),'CHANGE':(255,0,255),'PRINT':(255,120,0),'ANN':(0,0,255)}
def blend(gray,layers,alpha=.55):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);out=base.copy()
    for m,c in layers:
        col=np.zeros_like(base);col[:]=c;out[m]=cv2.addWeighted(base,1-alpha,col,alpha,0)[m]
    return out
for _,r in roster.iterrows():
    z=load_page(r);obs=z['obs'];seed=z['seed'];ring1=z['ring1'];ring2=z['ring2'];prepared=z['prepared'];corrected=z['corrected'];clean=z['clean'];changed=prepared&(np.abs(corrected.astype(np.int16)-obs.astype(np.int16))>=CHANGE_THRESHOLD)
    d=PAGES/r.page_key;d.mkdir(parents=True,exist_ok=False);cv2.imwrite(str(d/'observed.png'),obs);cv2.imwrite(str(d/'corrected.png'),corrected);cv2.imwrite(str(d/'authority_overlay.png'),blend(obs,[(seed,COL['SEED']),(ring1,COL['RING1']),(ring2,COL['RING2']),(changed,COL['CHANGE'])]))
    if clean is not None:
        tp=tolerant_clean_print(clean);ta=seed&(obs<DARK_THRESHOLD)&~tp;cv2.imwrite(str(d/'clean.png'),clean);cv2.imwrite(str(d/'truth_overlay_posthoc.png'),blend(obs,[(tp&prepared,COL['PRINT']),(ta,COL['ANN']),(changed,COL['CHANGE'])]))
    del z;gc.collect()

def sheet(rows,name,paired=False):
    if rows.empty:return
    files=[('observed.png','observed'),('corrected.png','corrected'),('authority_overlay.png','seed/rings/change')]
    if paired:files=[('observed.png','observed'),('clean.png','clean truth'),('corrected.png','corrected'),('truth_overlay_posthoc.png','truth ownership/change')]
    fig,axes=plt.subplots(len(rows),len(files),figsize=(5*len(files),5.3*len(rows)),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        for j,(fn,title) in enumerate(files):
            im=cv2.imread(str(PAGES/r.page_key/fn),cv2.IMREAD_UNCHANGED);im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB) if im is not None and im.ndim==3 else im;axes[i,j].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key}\n{title}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=125,bbox_inches='tight');plt.close(fig)
hitl=page_df[page_df.population=='HITL_ANNOTATED_ONLY'].sort_values('seed_whitened_observable',ascending=False).head(MAX_CONTACT_PAGES);paired=page_df[page_df.population=='PAIRED_DEVELOPMENT'].sort_values('seed_true_print_whitened',ascending=False).head(MAX_CONTACT_PAGES)
sheet(hitl,'worst_hitl_geometry_review',False);sheet(paired,'worst_paired_true_print_loss_review',True)

## 8. Conclusions and curated upload package

In [ ]:
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'pages_total':len(page_df),'pages_by_population':page_df.groupby('population').size().to_dict(),'exact_dilation2_all_pages':bool((checks.prepared_vs_exact_dilation2_xor==0).all()),'hitl_seed_observable_whitening':int(page_df.loc[page_df.population=='HITL_ANNOTATED_ONLY','seed_whitened_observable'].sum()),'hitl_collar_observable_whitening':int(page_df.loc[page_df.population=='HITL_ANNOTATED_ONLY','collar_whitened_observable'].sum()),'paired_seed_true_print_whitened':int(page_df.loc[page_df.population=='PAIRED_DEVELOPMENT','seed_true_print_whitened'].sum()),'paired_collar_true_print_whitened':int(page_df.loc[page_df.population=='PAIRED_DEVELOPMENT','collar_true_print_whitened'].sum()),'interpretation_warning':'HITL observable print-like evidence and paired clean-supported true print are different endpoints and are not pooled as truth.'}
atomic_json(OUT/'notebook74_combined_verdict.json',summary)
lines=['NOTEBOOK 74 COMBINED MASK AUTHORITY / ANNOTATION DISTANCE AUDIT','',f'Total pages: {summary["pages_total"]}',f'Pages by population: {summary["pages_by_population"]}',f'Exact dilation-2 lineage on all pages: {summary["exact_dilation2_all_pages"]}','','HITL OBSERVABLE ENDPOINT',f'Seed whitening: {summary["hitl_seed_observable_whitening"]}',f'Collar whitening: {summary["hitl_collar_observable_whitening"]}','','PAIRED CLEAN-TRUTH ENDPOINT',f'Seed genuine-print whitening: {summary["paired_seed_true_print_whitened"]}',f'Collar genuine-print whitening: {summary["paired_collar_true_print_whitened"]}','','WARNING',summary['interpretation_warning']]
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook74_combined_verdict.json',INPUTS/'combined_roster.csv',AUDIT/'input_resolution_and_geometry.csv',TABLES/'page_summary.csv',TABLES/'authority_band_summary.csv',TABLES/'authority_by_distance_pooled.csv',VISUAL/'authority_band_change_rates_by_cohort.png',VISUAL/'seed_vs_collar_by_cohort.png']
for p in [VISUAL/'worst_hitl_geometry_review.png',VISUAL/'worst_paired_true_print_loss_review.png']:
    if p.exists():required.append(p)
for p in required:require(p.exists(),f'Missing package output {p}');shutil.copy2(p,UPLOAD/p.name)
manifest=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Combined Notebook 74 audit. HITL outputs are observable-only diagnostics; paired outputs use post-hoc clean truth. Start with executive_summary.txt and the visuals.\n')
with zipfile.ZipFile(UPLOAD/'notebook74_combined_audit_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
print('Bundle:',UPLOAD/'notebook74_combined_audit_bundle.zip')

## Interpretation guardrail

The combined run is computationally convenient, but the cohorts have different evidence roles:

- HITL pages provide production-like, clean-free spatial diagnostics of the current V8.8 outputs.
- Paired development pages provide post-hoc clean-supported truth for genuine printed ink and annotation distance, using frozen V7.8 development outputs.

Compare geometry patterns across cohorts, but do not report one pooled “true print loss rate” across all pages. Only the paired cohort has clean truth.